# 04 · IDEAM: deforestación oficial, por celda y periodo

**Qué hace.** Baja las capas de cambio de la superficie cubierta por bosque natural del IDEAM (Sistema de
Monitoreo de Bosques y Carbono, SMByC) y cuenta, en cada celda de 5 km y en cada periodo, cuántas hectáreas
fueron deforestadas, cuántas siguieron siendo bosque, cuántas se regeneraron y cuántas no se pudieron
observar por nubes.

**Cuándo correrlo.** Después de `02_carga_hansen`. Para actualizar, una vez al año, cuando el IDEAM publica
el periodo nuevo: se agrega en la celda 1 y se vuelve a correr.

**Qué lee.** Las tablas `GRILLA_CELDAS_5KM` y `BOSQUE_BASE_HANSEN_CELDA_5KM`, y las capas del IDEAM, que se bajan de su servidor público
(unos 50 MB por periodo, sin cuenta ni llave).

**Qué escribe**, en `DATASET_DEFORESTACION`:

| Tabla | Qué es |
|---|---|
| `DEFORESTACION_IDEAM_CELDA_5KM_PERIODO` | Una fila por celda y periodo: hectáreas de cada clase de cambio y bosque natural al inicio y al final |

**Qué tienes que cambiar.** Nada: la celda 1 ya apunta al proyecto del Observatorio (`prueba-ofr`).

**Cuánto tarda.** Unos 10 minutos.

**Qué mide.** Es la única tabla que mide deforestación en sentido estricto: conversión de bosque natural a
otra cobertura, con la definición nacional de bosque. Es el insumo de la cifra oficial de Colombia, y esta
agregación la reproduce entre el 99,65 % y el 99,84 % según el periodo.

In [ ]:
# =====================================================================
# 1. CONFIGURACIÓN
#    Ya está llena para el proyecto del Observatorio (prueba-ofr). Solo
#    hay que cambiarla si las tablas se pasan a otro proyecto o dataset.
# =====================================================================
# --- Dónde están las tablas en BigQuery ------------------------------
# Proyecto de Google Cloud del Observatorio (Prueba-OFR). Es el id,
# no el nombre: el que aparece en la consola al elegir el proyecto.
PROYECTO = "prueba-ofr"

# Dataset con las tablas de deforestación que arman los notebooks 01
# a 06 (grilla, paneles y panel municipal).
DATASET_DEFORESTACION = "Cambio_Climatico"

# Ubicación de los datos. Debe ser la misma de las tablas del
# Observatorio (en BigQuery: abre una tabla > Detalles > Ubicación de
# los datos). Las del Observatorio están hoy en US.
UBICACION = "US"

# --- Para actualizar --------------------------------------------------
# Periodos que entran en la tabla. Cada uno compara dos imágenes anuales
# ("2024-2025" compara 2024 con 2025) y su cifra corresponde al año final.
IDEAM_PERIODOS = ["2020-2021", "2021-2022", "2022-2023", "2023-2024", "2024-2025"]

# Nombre exacto de cada capa en el servidor del IDEAM. El final del nombre
# (v8_260729) no sigue un patrón que se pueda adivinar. Cuando el IDEAM
# publique un periodo nuevo, busca el nombre del archivo .img en
#   https://bart.ideam.gov.co/smbyc/Cambio%20en%20la%20superficie%20cubierta%20por%20bosque%20natural/Capas/
# agrégalo aquí y agrega el periodo a IDEAM_PERIODOS.
CAPAS_IDEAM = {
    "2015-2016": "cambio_2015-2016_v7_170629.img",
    "2016-2017": "cambio_2016-2017_v7_180612.img",
    "2017-2018": "cambio_2017-2018_v8_190616.img",
    "2018-2019": "cambio_2018_2019_v8_200707.img",
    "2019-2020": "cambio_2019-2020_v8_210702.img",
    "2020-2021": "cambio_2020_2021_v8_220710.img",
    "2021-2022": "cambio_2021_2022_v8_230705.img",
    "2022-2023": "cambio_2022_2023_V8_240703.img",
    "2023-2024": "cambio_2023_2024_v8_250721.img",
    "2024-2025": "cambio_2024_2025_v8_260729.img",
}

# --- Del trabajo de grado (no se cambian) ------------------------------
BOSQUE_MINIMO_HA = 50     # celdas con menos bosque base quedan fuera, como en los otros paneles
RESOLUCION_M = 25         # mismo píxel que el notebook 02, para que las hectáreas sean comparables
BLOQUE_CELDAS = 20

# --- Solo si corres el notebook en tu computador ----------------------
# En BigQuery déjalo vacío. En un computador: la ruta al archivo .json
# de la llave de una cuenta de servicio con acceso a estas tablas.
RUTA_LLAVE_BIGQUERY = ""

# Carpeta donde se bajan los archivos mientras corre el notebook.
# Vacío: una carpeta temporal. Si ya los tienes bajados, apúntala ahí
# y no se vuelven a descargar.
CARPETA_DESCARGAS = ""

In [ ]:
# =====================================================================
# 2. INSTALAR PAQUETES
#    En BigQuery instala lo que le falte a la máquina; la primera vez
#    tarda uno o dos minutos. No hay que cambiar nada. En un computador
#    con el entorno del repositorio (requirements_local.txt) se puede
#    saltar.
#    Las versiones fijas son las que se usaron en el trabajo de grado,
#    para que los resultados salgan iguales.
# =====================================================================
%pip install --quiet rasterio==1.4.3 pandas-gbq

In [ ]:
# =====================================================================
# 3. CONEXIÓN CON BIGQUERY Y FUNCIONES DE APOYO
#    No hay que cambiar nada en esta celda. Se conecta a BigQuery y
#    define las funciones con que el resto del notebook lee y escribe
#    tablas. Es igual en todos los notebooks de esta carpeta.
# =====================================================================
import json
import re
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery

NOMBRE_NOTEBOOK = "04_carga_ideam_cambio"

# --- 3.1 ¿Quedó algo sin llenar en la celda 1? --------------------------
# Si queda algún <<LLENAR ...>>, el notebook se detiene aquí y dice cuál.
_pendientes = sorted(n for n, v in list(globals().items())
                     if n.isupper() and isinstance(v, str) and "<<LLENAR" in v)
if _pendientes:
    raise ValueError("Falta llenar en la celda 1: " + ", ".join(_pendientes))

# --- 3.2 Conexión ---------------------------------------------------------
# En BigQuery el notebook entra con la cuenta de quien lo corre y no
# necesita llave. La llave solo se usa si se corre en un computador.
if globals().get("RUTA_LLAVE_BIGQUERY"):
    from google.oauth2 import service_account
    client = bigquery.Client(
        project=PROYECTO,
        credentials=service_account.Credentials.from_service_account_file(RUTA_LLAVE_BIGQUERY))
else:
    client = bigquery.Client(project=PROYECTO)
print(f"Conectado a BigQuery: proyecto {PROYECTO}")

# Carpeta para los archivos que se descargan mientras corre el notebook.
# Es de paso: todo lo que importa termina en tablas de BigQuery.
_carpeta = globals().get("CARPETA_DESCARGAS") or ""
DIR_TRABAJO = Path(_carpeta) if _carpeta else Path(tempfile.gettempdir()) / "ofr_deforestacion"
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)


# --- 3.3 Nombres de tabla -----------------------------------------------
def nombre_completo(tabla, dataset=None):
    """'deforestacion_ideam_celda_5km_periodo' -> 'proyecto.dataset.DEFORESTACION_IDEAM_CELDA_5KM_PERIODO'.

    Las tablas que crean estos notebooks se nombran en mayúscula
    (GRILLA_CELDAS_5KM, DEFORESTACION_IDEAM_CELDA_5KM_PERIODO...). Si la tabla ya viene completa
    (proyecto.dataset.tabla, como las del Observatorio), se devuelve igual.
    Si no, se arma con PROYECTO y el dataset indicado; por defecto,
    DATASET_DEFORESTACION.
    """
    if tabla.count(".") == 2:
        return tabla
    return f"{PROYECTO}.{dataset or globals().get('DATASET_DEFORESTACION')}.{tabla.upper()}"


# --- 3.4 Leer -------------------------------------------------------------
_MARCA = re.compile(r"^(\[índice(?: sin nombre)?\] )?(?:Nombre original: «(.*?)»)?")


_LECTURA_RAPIDA = [True]


def leer_sql(sql):
    """Corre una consulta SQL en BigQuery y devuelve un DataFrame de pandas.

    Usa la lectura rápida de BigQuery (Storage Read API). Si no está
    disponible (por ejemplo, porque la cuenta no tiene el rol «BigQuery
    Read Session User»), sigue con la lectura normal, que da el mismo
    resultado pero tarda más en tablas grandes.
    """
    trabajo = client.query(sql)
    trabajo.result()                      # si la consulta tiene un error, sale aquí
    resultado = None
    if _LECTURA_RAPIDA[0]:
        try:
            resultado = trabajo.to_arrow()
        except Exception as e:
            _LECTURA_RAPIDA[0] = False
            print("Aviso: la lectura rápida de BigQuery no está disponible "
                  f"({type(e).__name__}); se usa la lectura normal.")
    if resultado is None:
        resultado = trabajo.to_arrow(create_bqstorage_client=False)
    return resultado.to_pandas(date_as_object=False, coerce_temporal_nanoseconds=True)


def leer_tabla(tabla, columnas=None, dataset=None, filtro=None):
    """Lee una tabla de BigQuery a pandas.

    columnas  lista de columnas a traer; por defecto, todas.
    filtro    condición SQL opcional, p. ej. "anio >= 2021".

    Si la tabla se guardó con guardar_tabla(), devuelve las filas en el
    mismo orden en que se guardaron, las columnas con su nombre original
    (tildes, espacios) y el índice.
    """
    destino = nombre_completo(tabla, dataset)
    esquema = client.get_table(destino).schema
    con_orden = any(c.name == "orden_fila" for c in esquema)
    seleccion = ", ".join(f"`{c}`" for c in columnas) if columnas else "*"
    df = leer_sql(f"SELECT {seleccion} FROM `{destino}`"
                  + (f" WHERE {filtro}" if filtro else "")
                  + (" ORDER BY orden_fila" if con_orden else ""))
    df = df.drop(columns=["orden_fila"], errors="ignore")

    originales, indice, sin_nombre = {}, [], set()
    for campo in esquema:
        marca = _MARCA.match(campo.description or "")
        if marca.group(2):
            originales[campo.name] = marca.group(2)
        if marca.group(1) and campo.name in df.columns:
            indice.append(campo.name)
            if "sin nombre" in marca.group(1):
                sin_nombre.add(campo.name)
    if indice:
        df = df.set_index(indice)
        df.index = df.index.set_names(
            [None if n in sin_nombre else originales.get(n, n) for n in indice])
    return df.rename(columns=originales)


# --- 3.5 Escribir -----------------------------------------------------------
_DATASETS_LISTOS = set()


def _nombre_para_bigquery(nombre, usados):
    """Nombre de columna que BigQuery acepta siempre: letras sin tilde,
    números y guion bajo. Los que ya cumplen se dejan igual."""
    texto = str(nombre)
    if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,299}", texto) and texto.lower() not in usados:
        usados.add(texto.lower())
        return texto
    limpio = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    limpio = re.sub(r"[^0-9A-Za-z]+", "_", limpio).strip("_").lower() or "columna"
    if limpio[0].isdigit():
        limpio = "c_" + limpio
    limpio, base, k = limpio[:280], limpio[:280], 2
    while limpio.lower() in usados:
        limpio, k = f"{base}_{k}", k + 1
    usados.add(limpio.lower())
    return limpio


def _tipos_para_bigquery(df):
    """Deja cada columna en un tipo que BigQuery reciba sin error:
    categorías a texto, enteros con vacíos a entero, mezclas a texto."""
    for c in df.columns:
        s = df[c]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype(object).where(s.notna(), None)
        if s.dtype == object:
            valores = s.dropna()
            tipos = {type(v) for v in valores}
            if not tipos or tipos <= {str}:
                pass
            elif all(isinstance(v, (bool, np.bool_)) for v in valores):
                s = s.astype("boolean")
            elif all(isinstance(v, (int, np.integer)) and not isinstance(v, bool) for v in valores):
                s = s.astype("Int64")
            elif all(isinstance(v, (int, float, np.integer, np.floating)) for v in valores):
                s = s.astype(float)
            else:
                s = s.map(lambda v: None if v is None or v is pd.NA
                          or (isinstance(v, float) and v != v) else str(v))
        df[c] = s
    return df


def guardar_tabla(df, tabla, descripcion, dataset=None, indice=False,
                  descripciones=None):
    """Escribe `df` en BigQuery. Si la tabla ya existe, la REEMPLAZA entera.

    tabla          nombre corto, p. ej. "deforestacion_ideam_celda_5km_periodo".
    descripcion    texto que se ve en BigQuery, en los detalles de la tabla.
    dataset        por defecto, DATASET_DEFORESTACION. Si no existe, se crea.
    indice         True para guardar también el índice de pandas como
                   columna(s); leer_tabla() lo recupera.
    descripciones  {columna: texto}, para documentar columnas en BigQuery.
    """
    dataset = dataset or globals().get("DATASET_DEFORESTACION")
    destino = nombre_completo(tabla, dataset)
    if dataset not in _DATASETS_LISTOS:
        ds = bigquery.Dataset(f"{PROYECTO}.{dataset}")
        ds.location = globals().get("UBICACION", "US")
        client.create_dataset(ds, exists_ok=True)
        _DATASETS_LISTOS.add(dataset)

    datos = df.copy()
    marcas = {}
    if indice:
        nombres_indice = list(datos.index.names)
        datos = datos.reset_index()
        for k, n in enumerate(nombres_indice):
            columna = datos.columns[k]
            marcas[columna] = "[índice sin nombre] " if n is None else "[índice] "
    else:
        datos = datos.reset_index(drop=True)
    if isinstance(datos.columns, pd.MultiIndex):
        datos.columns = [" | ".join(str(x) for x in c if str(x) != "") for c in datos.columns]

    descripciones = descripciones or {}
    usados, notas, columnas = set(), {}, []
    for c in datos.columns:
        nuevo = _nombre_para_bigquery(c, usados)
        texto = marcas.get(c, "")
        if nuevo != str(c):
            texto += f"Nombre original: «{c}»"
        if descripciones.get(c):
            texto += (". " if texto and not texto.endswith("] ") else "") + descripciones[c]
        notas[nuevo] = texto[:1024]
        columnas.append(nuevo)
    datos.columns = columnas
    datos = _tipos_para_bigquery(datos)
    # BigQuery no guarda el orden de las filas, y algunos cálculos (el
    # muestreo del random forest, por ejemplo) dependen de él. Esta
    # columna lo conserva; leer_tabla() la usa y la quita.
    datos["orden_fila"] = np.arange(len(datos), dtype="int64")
    notas["orden_fila"] = ("Posición de la fila en la tabla original. Sirve para leerla "
                           "en el mismo orden: ORDER BY orden_fila.")

    trabajo = client.load_table_from_dataframe(
        datos, destino,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    trabajo.result()

    t = client.get_table(destino)
    t.description = descripcion
    t.schema = [bigquery.SchemaField(f.name, f.field_type, mode=f.mode,
                                     description=notas.get(f.name) or None,
                                     fields=f.fields) for f in t.schema]
    client.update_table(t, ["description", "schema"])
    print(f"Guardada {destino}: {t.num_rows:,} filas, {len(t.schema)} columnas")

## 4. La grilla y el bosque base

In [ ]:
LADO_CELDA_M = 5_000          # los de la grilla (notebook 01)
CRS_GRILLA = "EPSG:3116"
AREA_PX_HA = RESOLUCION_M ** 2 / 10_000

grilla = leer_tabla("grilla_celdas_5km")
bosque = leer_tabla("bosque_base_hansen_celda_5km")
if not grilla["cell_id"].equals(bosque["cell_id"]):
    raise RuntimeError("bosque_base_hansen_celda_5km no corresponde a esta grilla: vuelve a correr el notebook 02.")
mapa_celda = {(int(a), int(b)): i for i, (a, b) in enumerate(zip(grilla["ix"], grilla["iy"]))}
print(f"Grilla: {len(grilla):,} celdas")

## 5. Descargar las capas del IDEAM

Son archivos ERDAS Imagine (`.img`), que se leen directamente, ya en la proyección de la grilla y a unos
30 m por píxel. Los `.rrd` que los acompañan en el servidor son solo pirámides para ver el mapa y no se
bajan.

**La dirección tiene que ser `https`.** El servidor del IDEAM tiene cerrado el puerto 80: con `http` no
responde con un error, sino que se queda esperando hasta agotar el tiempo.

In [ ]:
import requests

URL_IDEAM = ("https://bart.ideam.gov.co/smbyc/"
             "Cambio%20en%20la%20superficie%20cubierta%20por%20bosque%20natural/Capas")
desconocidos = [p for p in IDEAM_PERIODOS if p not in CAPAS_IDEAM]
if desconocidos:
    raise ValueError(f"Falta el nombre del archivo de {desconocidos} en CAPAS_IDEAM (celda 1).")


def descargar_capa(periodo, intentos=3):
    """Baja la capa de un periodo, salvo que ya esté. Devuelve su ruta."""
    destino = DIR_TRABAJO / f"cambio_{periodo}.img"
    if destino.exists() and destino.stat().st_size > 0:
        print(f"  {periodo}: ya está ({destino.stat().st_size / 1e6:.0f} MB)")
        return destino
    temporal = destino.with_suffix(".parcial")
    for intento in range(1, intentos + 1):
        try:
            with requests.get(f"{URL_IDEAM}/{CAPAS_IDEAM[periodo]}", stream=True, timeout=300) as r:
                r.raise_for_status()
                total = int(r.headers.get("Content-Length", 0))
                bajado = 0
                with open(temporal, "wb") as f:
                    for trozo in r.iter_content(chunk_size=1 << 20):
                        f.write(trozo)
                        bajado += len(trozo)
            if total and bajado != total:
                raise IOError(f"llegó incompleto: {bajado} de {total} bytes")
            temporal.replace(destino)
            print(f"  {periodo}: listo ({bajado / 1e6:.0f} MB)")
            return destino
        except Exception as e:
            print(f"  {periodo}: intento {intento} de {intentos} falló ({e})")
            temporal.unlink(missing_ok=True)
    raise RuntimeError(f"No se pudo bajar la capa {periodo}. Vuelve a correr esta celda.")


RUTAS = {p: descargar_capa(p) for p in IDEAM_PERIODOS}

## 6. Cómo se reparte un ráster entre las celdas

Estas funciones son el corazón geométrico de los notebooks 02 y 04, y son idénticas en los dos. Si cada
tabla repartiera los píxeles a su manera, una diferencia de medio píxel en el borde de una celda haría que
dejaran de ser comparables.

1. **Bloques.** La grilla se parte en bloques de 20 × 20 celdas, alineados con el origen. Solo se usan los
   bloques que tienen alguna celda de la grilla.
2. **Ráster de destino.** Para cada bloque se define un ráster de 25 m en la proyección de la grilla, que
   encaja exacto con los bordes de las celdas: 20 celdas × 5.000 m / 25 m = 4.000 píxeles, sin residuo.
3. **Reproyección.** Cada capa se reproyecta sobre ese ráster con vecino más cercano, porque las capas son
   categorías (porcentaje de dosel, año de pérdida, clase de cambio) y promediar categorías no tiene
   sentido.
4. **Celda de cada píxel.** Como el ráster de destino está en la proyección de la grilla, la celda de cada
   píxel es una división: `floor(x / 5000)`. Los bloques no se traslapan, así que los conteos de distintos
   bloques se suman sin contar nada dos veces.

In [ ]:
import rasterio
from rasterio.transform import Affine
from rasterio.warp import Resampling, reproject


def bloques(grilla):
    """Bloques (ix0, iy0, ix1, iy1), con ix1 e iy1 excluidos, que cubren las celdas de la grilla."""
    paso = BLOQUE_CELDAS
    ocupados = set(zip(grilla["ix"] // paso, grilla["iy"] // paso))
    return sorted((bx * paso, by * paso, (bx + 1) * paso, (by + 1) * paso) for bx, by in ocupados)


def perfil_bloque(ix0, iy0, ix1, iy1):
    """Ráster de destino del bloque: transformación, proyección y tamaño."""
    x_min, x_max = ix0 * LADO_CELDA_M, ix1 * LADO_CELDA_M
    y_min, y_max = iy0 * LADO_CELDA_M, iy1 * LADO_CELDA_M
    ancho = int(round((x_max - x_min) / RESOLUCION_M))
    alto = int(round((y_max - y_min) / RESOLUCION_M))
    # Origen arriba a la izquierda; -RESOLUCION_M porque las filas bajan y la Y sube.
    transform = Affine(RESOLUCION_M, 0, x_min, 0, -RESOLUCION_M, y_max)
    return {"transform": transform, "crs": CRS_GRILLA, "height": alto, "width": ancho}


def indice_celdas_bloque(perfil, mapa_celda):
    """Para cada píxel del bloque, su fila en la tabla de la grilla (-1 si cae fuera de la grilla).

    Como el ráster no está rotado, la X de un píxel depende solo de su
    columna y la Y solo de su fila: basta resolver las celdas de una fila
    y una columna y combinarlas, en vez de recorrer 16 millones de píxeles.
    Se usa el CENTRO del píxel (+0,5) para decidir su celda.
    """
    tr = perfil["transform"]
    cols = np.arange(perfil["width"]) + 0.5
    filas = np.arange(perfil["height"]) + 0.5
    ix_col = np.floor((tr.c + tr.a * cols) / LADO_CELDA_M).astype(np.int64)
    iy_fila = np.floor((tr.f + tr.e * filas) / LADO_CELDA_M).astype(np.int64)
    ux, pos_x = np.unique(ix_col, return_inverse=True)
    uy, pos_y = np.unique(iy_fila, return_inverse=True)
    tabla = np.array([[mapa_celda.get((int(a), int(b)), -1) for a in ux] for b in uy], dtype=np.int32)
    return tabla[pos_y[:, None], pos_x[None, :]]


def reproyectar_sobre_bloque(rutas, perfil, dtype=np.uint8):
    """Reproyecta uno o varios rásteres sobre el bloque y los combina.

    Donde un archivo no tiene datos, reproject() deja 0; el máximo entre
    archivos se queda con el valor real del que sí cubre el píxel. Eso
    supone que 0 es "sin dato", y así es en Hansen (0 = sin pérdida) y en
    el IDEAM (las clases van de 1 a 5).
    """
    salida = np.zeros((perfil["height"], perfil["width"]), dtype=dtype)
    for ruta in rutas:
        with rasterio.open(ruta) as src:
            trozo = np.zeros_like(salida)
            reproject(source=rasterio.band(src, 1), destination=trozo,
                      dst_transform=perfil["transform"], dst_crs=perfil["crs"],
                      resampling=Resampling.nearest, src_nodata=None, dst_nodata=0)
        salida = np.maximum(salida, trozo)
    return salida


def contar_por_celda_y_clase(valores, idx, n_celdas, n_clases):
    """Matriz (celdas x clases) con el número de píxeles de cada clase en cada celda.

    Un solo np.bincount sobre el índice combinado celda * n_clases + clase,
    en vez de uno por clase. Se descartan los píxeles fuera de la grilla
    (idx = -1) y los de clase 0 (sin dato).
    """
    plano_idx = idx.ravel()
    plano_val = valores.ravel().astype(np.int64)
    ok = (plano_idx >= 0) & (plano_val > 0) & (plano_val < n_clases)
    conteo = np.bincount(plano_idx[ok] * n_clases + plano_val[ok], minlength=n_celdas * n_clases)
    return conteo[:n_celdas * n_clases].reshape(n_celdas, n_clases)

## 7. Hectáreas de cada clase, celda por celda

La leyenda de las capas (archivo `Contenido_Cambio.txt` del servidor del IDEAM):

| Clase | Qué es | Columna |
|---|---|---|
| 1 | Bosque estable | `bosque_estable_ha` |
| 2 | Deforestación | `def_ha` |
| 3 | Sin información (nubes) | `sin_info_ha` |
| 4 | Regeneración | `regeneracion_ha` |
| 5 | No bosque estable | `no_bosque_ha` |

Las capas ya vienen en la proyección de la grilla, pero a 30 m y con su propio origen. Se reproyectan
igual al píxel de 25 m del notebook 02, para que las hectáreas del IDEAM y las de Hansen salgan de la misma
rejilla y se puedan comparar celda por celda.

Todos los periodos se procesan dentro del mismo recorrido de bloques: la celda de cada píxel depende solo
del bloque, así que se calcula una vez y sirve para todos los periodos.

In [ ]:
import time

CLASES = {1: "bosque_estable_ha", 2: "def_ha", 3: "sin_info_ha", 4: "regeneracion_ha", 5: "no_bosque_ha"}
N_CLASES = 6                  # 0 a 5; el 0 es fuera de cobertura y se descarta
n_celdas = len(grilla)
totales = {p: np.zeros((n_celdas, N_CLASES), dtype=np.float64) for p in IDEAM_PERIODOS}
lista_bloques = bloques(grilla)
print(f"Recorriendo {len(lista_bloques)} bloques con {len(IDEAM_PERIODOS)} periodos...")

inicio = time.time()
for k, (ix0, iy0, ix1, iy1) in enumerate(lista_bloques, 1):
    perfil = perfil_bloque(ix0, iy0, ix1, iy1)
    idx = indice_celdas_bloque(perfil, mapa_celda)          # una vez por bloque
    for periodo in IDEAM_PERIODOS:
        clases = reproyectar_sobre_bloque([RUTAS[periodo]], perfil)
        totales[periodo] += contar_por_celda_y_clase(clases, idx, n_celdas, N_CLASES) * AREA_PX_HA
    if k % 20 == 0 or k == len(lista_bloques):
        acumulado = sum(m[:, 2].sum() for m in totales.values())
        print(f"  [{k}/{len(lista_bloques)}] deforestación acumulada, todos los periodos: "
              f"{acumulado:,.0f} ha · {time.time() - inicio:.0f} s")

## 8. La tabla

Una fila por celda y periodo. Además de las cinco clases, lleva dos medidas del bosque natural, que **no se
deben confundir**:

- `bosque_ideam_ha` = bosque estable + deforestación: el bosque natural al **inicio** del periodo. Es el
  denominador correcto de la tasa de esta tabla, porque `def_ha` solo cuenta bosque natural.
- `bosque_ideam_fin_ha` = bosque estable + regeneración: el bosque al **final** del periodo. Dice cuánto bosque
  queda; no sirve para normalizar la deforestación del periodo.

`bosque_base_ha` es el bosque de Hansen, el de las otras tablas. Es un 31 % mayor que el del IDEAM porque
cuenta cualquier píxel con dosel, plantaciones incluidas. Dividir `def_ha` entre él subestima la tasa.

Se quedan las celdas con al menos 50 ha de bosque base, igual que en las otras tablas, para que todas cubran
las mismas celdas.

In [ ]:
filas = []
for periodo in IDEAM_PERIODOS:
    matriz = totales[periodo]
    bloque = pd.DataFrame({"cell_id": grilla["cell_id"].values, "periodo": periodo})
    for codigo, columna in CLASES.items():
        bloque[columna] = matriz[:, codigo]
    filas.append(bloque)
panel_ideam = pd.concat(filas, ignore_index=True)
panel_ideam["bosque_ideam_ha"] = panel_ideam["bosque_estable_ha"] + panel_ideam["def_ha"]
panel_ideam["bosque_ideam_fin_ha"] = panel_ideam["bosque_estable_ha"] + panel_ideam["regeneracion_ha"]

meta = grilla[["cell_id", "lon", "lat", "departamento"]].copy()
meta["bosque_base_ha"] = bosque["bosque_base_ha"].values
panel_ideam = panel_ideam.merge(meta, on="cell_id", how="left")
panel_ideam = panel_ideam[panel_ideam["bosque_base_ha"] >= BOSQUE_MINIMO_HA].copy()
panel_ideam = panel_ideam.merge(grilla[["cell_id", "cod_dane", "municipio"]], on="cell_id", how="left")
panel_ideam = panel_ideam.sort_values(["cell_id", "periodo"]).reset_index(drop=True)

print(f"deforestacion_ideam_celda_5km_periodo: {len(panel_ideam):,} filas = {panel_ideam.cell_id.nunique():,} celdas x "
      f"{panel_ideam.periodo.nunique()} periodos")
(panel_ideam.groupby("periodo")[["def_ha", "sin_info_ha", "bosque_ideam_ha", "bosque_ideam_fin_ha"]]
            .sum().round(0))

## 9. Guardar en BigQuery

In [ ]:
guardar_tabla(
    panel_ideam, "deforestacion_ideam_celda_5km_periodo",
    "Deforestación oficial por celda de 5 km y periodo, según las capas de cambio de la superficie cubierta "
    "por bosque natural del IDEAM (SMByC). Cada periodo compara dos años y su cifra corresponde al año "
    f"final. Solo celdas con al menos {BOSQUE_MINIMO_HA} ha de bosque base.",
    descripciones={
        "cell_id": "Identificador de la celda (tabla GRILLA_CELDAS_5KM).",
        "periodo": "Transición entre dos años, p. ej. 2020-2021; la cifra corresponde al año final.",
        "bosque_estable_ha": "Hectáreas que siguieron siendo bosque natural (clase 1).",
        "def_ha": "Hectáreas de bosque natural deforestadas en el periodo (clase 2).",
        "sin_info_ha": "Hectáreas que no se pudieron observar, sobre todo por nubes (clase 3). "
                       "Una celda con mucha superficie aquí tiene un dato de deforestación poco confiable.",
        "regeneracion_ha": "Hectáreas que pasaron a ser bosque (clase 4).",
        "no_bosque_ha": "Hectáreas que no eran ni son bosque (clase 5).",
        "bosque_ideam_ha": "Bosque natural al inicio del periodo: bosque estable + deforestación. "
                           "Denominador de la tasa de esta tabla.",
        "bosque_ideam_fin_ha": "Bosque natural al final del periodo: bosque estable + regeneración.",
        "lon": "Longitud del centro de la celda.",
        "lat": "Latitud del centro de la celda.",
        "departamento": "Departamento del centro de la celda.",
        "bosque_base_ha": "Bosque según Hansen (tabla BOSQUE_BASE_HANSEN_CELDA_5KM); no es el denominador de esta tabla.",
        "cod_dane": "Código DANE del municipio del centro de la celda.",
        "municipio": "Nombre del municipio, solo para leer.",
    })

Listo. Sigue `05_carga_ideam_dtd`.